Selecting File

In [93]:
# https://stackoverflow.com/questions/3579568/choosing-a-file-in-python-with-simple-dialog

from tkinter import Tk     # from tkinter import Tk for Python 3.x
from tkinter.filedialog import askopenfilename

Tk().withdraw() # we don't want a full GUI, so keep the root window from appearing
filename = askopenfilename() # show an "Open" dialog box and return the path to the selected file
print(filename)

C:/Users/User/Downloads/adv_protocol_three-prob-2.log


Open content

In [94]:
# from my Advent of Code 2024 codes

import json

with open(filename, "r", encoding="utf-8") as file:
    content = file.read()

# 읽은 내용 출력
print(content)

content_list = []
for line_number, line in enumerate(content.splitlines(), start=1):
    if not line.strip():
        continue
    try:
        message = json.loads(line)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc
    if not isinstance(message, dict):
        raise ValueError(f"Line {line_number} must contain a JSON object")
    content_list.append(message)

print(content_list)

{"opcode": 0, "type": "DH"}
{"opcode": 1, "type": "DH", "public": 67, "parameter": {"p": 467, "g": 386}}
{"opcode": 1, "type": "DH", "public": 74}
{"opcode": 2, "type": "AES", "encryption": "M8cG9VOKSsm7BaH1ehELsw=="}
{"opcode": 2, "type": "AES", "encryption": "YSaVAJALeVQRutoR4h+rEg=="}

[{'opcode': 0, 'type': 'DH'}, {'opcode': 1, 'type': 'DH', 'public': 67, 'parameter': {'p': 467, 'g': 386}}, {'opcode': 1, 'type': 'DH', 'public': 74}, {'opcode': 2, 'type': 'AES', 'encryption': 'M8cG9VOKSsm7BaH1ehELsw=='}, {'opcode': 2, 'type': 'AES', 'encryption': 'YSaVAJALeVQRutoR4h+rEg=='}]


Checks whether given json follows the example format

In [95]:
def line_validity_check(target_list: list, line_number: int, opcode: int, typ: str, additional_keys: list = []): # type is a function so could not use exact word
    error_list = []
    target = target_list[line_number]
    
    keys = target.keys()
    
    if "opcode" in keys:
        if target["opcode"] != opcode:
            error_list.append(f"unpredicted opcode (!= {opcode})")
    else:
        error_list.append("missing opcode")
    if "type" in keys:
        if target["type"] != typ:
            error_list.append(f"unpredicted type (!= {typ})")
    else:
        error_list.append("missing type")
    
    for key in additional_keys:
        if key not in keys:
            error_list.append(f"missing {key}")
    
    # no need to handle when keys not mentioned in additional_keys are in target ig
    
    return error_list

In [96]:
def json_validity_check(target: list) -> bool:
    if len(target) < 4 or not target:
        print("Incomplete log: expected handshake and encrypted messages")
        return False

    protocol = target[0].get("type")
    if protocol == "DH":
        expected = [(0, "DH", ()), (1, "DH", ("public", "parameter")), (1, "DH", ("public",))]
        aes_start = 3
    elif protocol == "RSA":
        expected = [(0, "RSA", ()), (1, "RSA", ("public", "parameter")), (2, "RSA", ("encrypted_key",))]
        aes_start = 3
    else:
        print(f"Unsupported protocol: {protocol!r}")
        return False

    errors = []
    for index, (opcode, message_type, required_keys) in enumerate(expected):
        line_errors = line_validity_check(target, index, opcode, message_type, required_keys)
        if index == 1:
            parameter = target[index].get("parameter")
            expected_parameters = ("p", "g") if protocol == "DH" else ("n",)
            if not isinstance(parameter, dict) or not all(key in parameter for key in expected_parameters):
                line_errors.append(f"parameter must contain {', '.join(expected_parameters)}")
        if line_errors:
            errors.append((index, target[index], line_errors))

    for index in range(aes_start, len(target)):
        line_errors = line_validity_check(target, index, 2, "AES", ("encryption",))
        if line_errors:
            errors.append((index, target[index], line_errors))

    for index, message, line_errors in errors:
        print(f"error in line {index}\n{message}\n{', '.join(line_errors)}")
    return not errors

In [97]:
json_validity_check(content_list)

True

Find key values

In [98]:
# written by parksh5295

def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

In [99]:
def find_pq(n: int, p_min: int = 400, p_max: int = 500) -> list:
    for p in range(p_min, p_max + 1):
        if is_prime(p):
            if n % p == 0:
                q = int(n / p)
                if is_prime(q):
                    return [p, q] # few of handlings seems unnessary, but just in case
    print("could not find p and q in given range")
    return False

In [100]:
def is_relative_prime(a: int, b: int):
    '''RSA: checks whether [a] and [b] are relatively prime to each other
    returns True when [a] and [b] are relative prime, else return False
    requires function [is_prime]'''
    
    if a <= 0 or b <= 0:
        print('Issue from [is_relative_prime] function: more than one value is not positive')
        return False

    high = max(a, b)
    low = min(a, b)
    
    if low == 1 or high == low:
        return True
    
    for i in range(2, low + 1):
        if is_prime(i):
            if low % i == 0 and high % i == 0:
                return False
            
    return True

def d_from(e: int, phin: int):
    '''RSA: calculates inverse of [e] in modular of [phin]
    checks whether [e] and [phin] are relative prime before calculating, therefore requiring function [is_relative_prime]'''
    if not is_relative_prime(e, phin):
        print("Issue from [d_from] function: e and phin are not relative prime")
        return -1
    for i in range(1, phin + 1): # planed to optimize this part with EEA, but cancelled
        if e * i % phin == 1:
            return i
    return -2 #this should never happen

In [101]:
protocol = content_list[0]["type"]
aes_key = None

if protocol == "RSA":
    n = content_list[1]["parameter"]["n"]
    factors = find_pq(n)
    if not factors:
        raise ValueError(f"Could not factor RSA modulus n={n} in the configured range")
    p, q = factors
    e = content_list[1]["public"]
    phi_n = (p - 1) * (q - 1)
    d = pow(e, -1, phi_n)
    encrypted_key = content_list[2]["encrypted_key"]
    decrypted_key = bytes(pow(value, d, n) for value in encrypted_key)
    aes_key = decrypted_key
    print(f"Detected protocol: RSA (p={p}, q={q}, e={e}, n={n})")
    print(f"Recovered AES key ({len(aes_key)} bytes): {aes_key!r}")
elif protocol == "DH":
    parameters = content_list[1]["parameter"]
    p, g = parameters["p"], parameters["g"]
    bob_public = content_list[1]["public"]
    alice_public = content_list[2]["public"]
    alice_private = next(
        (candidate for candidate in range(2, p) if pow(g, candidate, p) == alice_public),
        None,
    )
    if alice_private is None:
        raise ValueError("Could not recover a DH private value from the logged toy parameters")
    shared_secret = pow(bob_public, alice_private, p)
    aes_key = shared_secret.to_bytes(2, "big") * 16
    print(f"Detected protocol: DH (p={p}, g={g})")
    print(f"Recovered shared secret: {shared_secret}")
    print(f"Derived AES key ({len(aes_key)} bytes): {aes_key!r}")
else:
    raise ValueError(f"Unsupported protocol: {protocol!r}")

Detected protocol: DH (p=467, g=386)
Recovered shared secret: 122
Derived AES key (32 bytes): b'\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z\x00z'


Decryptes key list

In [102]:
def rsa_decryption(encrypted_message, d, n):
    """Decrypt one RSA integer with private exponent d and modulus n."""
    return pow(encrypted_message, d, n)

In [103]:
if protocol == "RSA":
    print("RSA encrypted AES key recovered from the encrypted_key field.")
else:
    print("DH AES key derived from the logged public values and toy parameters.")

DH AES key derived from the logged public values and toy parameters.


Finding the message

In [104]:
# from example

from Crypto.Cipher import AES
import base64

def decrypt_aes(key, encrypted):
    """Decrypt a Base64-encoded AES-ECB message and remove PKCS#7 padding."""
    decrypted = AES.new(key, AES.MODE_ECB).decrypt(base64.b64decode(encrypted))
    pad_length = decrypted[-1]
    if not 1 <= pad_length <= AES.block_size or decrypted[-pad_length:] != bytes([pad_length]) * pad_length:
        raise ValueError("Invalid AES padding")
    return decrypted[:-pad_length].decode("utf-8")

In [105]:
if not json_validity_check(content_list):
    raise ValueError("The selected log does not match the supported protocol format")

for message_number, message in enumerate(content_list[3:], start=4):
    print(f"Message {message_number} ({protocol}): {decrypt_aes(aes_key, message['encryption'])}")

Message 4 (DH): YOUR IDOL
Message 5 (DH): rumi ZOEY mira
